# Qwen2.5-VL + LoRA inference on local RAISE images

This notebook tests the trained Qwen2.5-VL LoRA adapter on local RAISE `.NEF` images. RAISE images are treated as real images, so the main metric is the false positive rate: real images predicted as fake.


In [ ]:
# Optional. Keep False unless packages are missing.
INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import subprocess, sys
    packages = [
        "transformers>=4.51.0",
        "peft>=0.12.0",
        "accelerate",
        "bitsandbytes",
        "qwen-vl-utils",
        "rawpy",
        "pillow",
        "pandas",
        "numpy",
        "tqdm",
        "matplotlib",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


In [ ]:
from pathlib import Path
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from peft import PeftModel
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", message="Palette images with Transparency")


In [ ]:
# -----------------------------
# Config
# -----------------------------
DATA_ROOT = Path(r"D:\LLM\LVLM\crawl\RAISE_200_NEF")
ADAPTER_DIR = Path(r"D:\LLM\LVLM\MLLM_detect_fake_image\Output_bakup\Qwen-7B\20260910_003026\adapter")
OUTPUT_ROOT = Path(r"D:\LLM\LVLM\MLLM_detect_fake_image\HoangHa_Code\outputs\qwen25vl_lora_raise_local")

BASE_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
PROMPT_TEMPLATE = "Look at the image and decide whether it is real or AI-generated. Answer with exactly one word: real or fake."
CANDIDATE_TEXTS = ["real", "fake"]
NORMALIZE_CANDIDATE_LOGPROB = True

# None = use all images. RAISE_200_NEF currently has 200 images.
MAX_IMAGES = None
RANDOM_SEED = 42
BATCH_SIZE = 1
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Qwen2.5-VL-7B needs GPU. 4-bit reduces VRAM usage.
LOAD_IN_4BIT = True
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".nef"}

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = OUTPUT_ROOT / RUN_ID
METRICS_DIR = RUN_DIR / "metrics"
PREDICTIONS_DIR = RUN_DIR / "predictions"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print("DATA_ROOT =", DATA_ROOT)
print("ADAPTER_DIR =", ADAPTER_DIR)
print("RUN_DIR =", RUN_DIR)
print("DEVICE =", DEVICE)


In [ ]:
def save_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def find_image_paths(root: Path, max_images=None, seed: int = 42) -> list[Path]:
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")
    paths = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if max_images is not None and len(paths) > max_images:
        rng = random.Random(seed)
        paths = sorted(rng.sample(paths, max_images))
    return paths


def load_rgb_image(path: Path) -> Image.Image:
    if path.suffix.lower() == ".nef":
        import rawpy
        with rawpy.imread(str(path)) as raw:
            rgb = raw.postprocess(use_camera_wb=True, no_auto_bright=False, output_bps=8)
        return Image.fromarray(rgb).convert("RGB")
    return Image.open(path).convert("RGB")


paths = find_image_paths(DATA_ROOT, MAX_IMAGES, RANDOM_SEED)
meta_df = pd.DataFrame([
    {
        "sample_id": idx,
        "image_path": str(path),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "parent_folder": path.parent.name,
        "label": 0,
        "label_name": "real",
        "dataset_tag": "raise_200_nef_real_only",
    }
    for idx, path in enumerate(paths)
])

meta_df.to_csv(METRICS_DIR / "raise_selected_images.csv", index=False)
meta_df.groupby("extension").size().reset_index(name="num_samples").to_csv(METRICS_DIR / "raise_extension_counts.csv", index=False)
meta_df.groupby("parent_folder").size().reset_index(name="num_samples").to_csv(METRICS_DIR / "raise_parent_folder_counts.csv", index=False)

print("num_images =", len(paths))
display(meta_df.head())


In [ ]:
def load_model_and_processor():
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f"LoRA adapter folder does not exist: {ADAPTER_DIR}")
    if DEVICE != "cuda":
        raise RuntimeError("Qwen2.5-VL-7B inference is not practical on CPU. Please run this notebook on a CUDA GPU machine.")

    try:
        processor = AutoProcessor.from_pretrained(ADAPTER_DIR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, trust_remote_code=True)
    except Exception:
        processor = AutoProcessor.from_pretrained(BASE_MODEL_NAME, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, trust_remote_code=True)

    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_NAME,
        device_map="auto",
        torch_dtype=DTYPE,
        quantization_config=quantization_config,
        trust_remote_code=True,
    )
    base_model.config.use_cache = True
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
    model.eval()
    return model, processor


model, processor = load_model_and_processor()
for text in CANDIDATE_TEXTS:
    token_ids = processor.tokenizer.encode(text, add_special_tokens=False)
    print(repr(text), token_ids, "num_tokens=", len(token_ids))


In [ ]:
def make_user_messages(image: Image.Image):
    return [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT_TEMPLATE}]}]


def make_full_messages(image: Image.Image, answer: str):
    return make_user_messages(image) + [{"role": "assistant", "content": [{"type": "text", "text": answer}]}]


def move_inputs_to_device(inputs: dict):
    return {key: value.to(DEVICE) if torch.is_tensor(value) else value for key, value in inputs.items()}


def sequence_logprob_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].float()
    shift_labels = labels[:, 1:]
    mask = shift_labels.ne(-100)
    safe_labels = shift_labels.masked_fill(~mask, 0)
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_scores = log_probs.gather(dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1) * mask
    scores = token_scores.sum(dim=-1)
    if NORMALIZE_CANDIDATE_LOGPROB:
        scores = scores / mask.sum(dim=-1).clamp_min(1)
    return scores


@torch.no_grad()
def score_candidate_batch(images: list[Image.Image], candidate_text: str) -> np.ndarray:
    full_texts = []
    prompt_lengths = []
    for image in images:
        prompt_text = processor.apply_chat_template(make_user_messages(image), tokenize=False, add_generation_prompt=True)
        full_text = processor.apply_chat_template(make_full_messages(image, candidate_text), tokenize=False, add_generation_prompt=False)
        prompt_inputs = processor(text=[prompt_text], images=[image], padding=False, return_tensors="pt")
        prompt_lengths.append(prompt_inputs["input_ids"].shape[1])
        full_texts.append(full_text)

    inputs = processor(text=full_texts, images=images, padding=True, return_tensors="pt")
    candidate_labels = inputs["input_ids"].clone()
    for row_idx, prompt_len in enumerate(prompt_lengths):
        candidate_labels[row_idx, :prompt_len] = -100
    candidate_labels[candidate_labels == processor.tokenizer.pad_token_id] = -100

    inputs = move_inputs_to_device(inputs)
    candidate_labels = candidate_labels.to(DEVICE)
    outputs = model(**inputs)
    return sequence_logprob_from_logits(outputs.logits, candidate_labels).detach().cpu().numpy()


def iter_batches(items, batch_size: int):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


In [ ]:
def predict_raise_real(paths: list[Path]) -> pd.DataFrame:
    rows = []
    errors = []
    pbar = tqdm(total=len(paths), desc="raise_real_only")

    for batch_paths in iter_batches(paths, BATCH_SIZE):
        images = []
        valid_paths = []
        for path in batch_paths:
            try:
                image = load_rgb_image(path)
                images.append(image)
                valid_paths.append(path)
            except Exception as exc:
                errors.append({"image_path": str(path), "error": repr(exc)})
                pbar.update(1)

        if not images:
            continue

        real_scores = score_candidate_batch(images, "real")
        fake_scores = score_candidate_batch(images, "fake")
        probs = torch.softmax(torch.tensor(np.stack([real_scores, fake_scores], axis=1)), dim=-1).numpy()

        for path, image, fake_prob in zip(valid_paths, images, probs[:, 1]):
            rows.append({
                "sample_id": len(rows),
                "image_path": str(path),
                "file_name": path.name,
                "extension": path.suffix.lower(),
                "parent_folder": path.parent.name,
                "width": image.width,
                "height": image.height,
                "label": 0,
                "label_name": "real",
                "fake_probability": float(fake_prob),
                "predicted_label": int(fake_prob >= 0.5),
                "predicted_label_name": "fake" if fake_prob >= 0.5 else "real",
                "model_name": "qwen25vl_lora_word_label",
                "dataset_tag": "raise_200_nef_real_only",
                "adapter_dir": str(ADAPTER_DIR),
            })
        pbar.update(len(images))

    pbar.close()
    if errors:
        pd.DataFrame(errors).to_csv(METRICS_DIR / "raise_read_errors.csv", index=False)
    return pd.DataFrame(rows)


pred_df = predict_raise_real(paths)
pred_df.to_csv(PREDICTIONS_DIR / "raise_real_predictions.csv", index=False)
display(pred_df.head())


In [ ]:
def summarize_real_only(df: pd.DataFrame, threshold: float = 0.5) -> dict:
    fake_probs = df["fake_probability"].to_numpy(dtype=float)
    pred_fake = fake_probs >= threshold
    num_samples = int(len(df))
    num_pred_fake = int(pred_fake.sum())
    num_pred_real = int(num_samples - num_pred_fake)
    return {
        "num_samples": num_samples,
        "threshold": float(threshold),
        "num_pred_real": num_pred_real,
        "num_pred_fake": num_pred_fake,
        "real_accuracy": float(num_pred_real / num_samples) if num_samples else 0.0,
        "false_positive_rate": float(num_pred_fake / num_samples) if num_samples else 0.0,
        "mean_fake_probability": float(np.mean(fake_probs)) if num_samples else None,
        "median_fake_probability": float(np.median(fake_probs)) if num_samples else None,
        "p90_fake_probability": float(np.quantile(fake_probs, 0.90)) if num_samples else None,
        "p95_fake_probability": float(np.quantile(fake_probs, 0.95)) if num_samples else None,
        "p99_fake_probability": float(np.quantile(fake_probs, 0.99)) if num_samples else None,
        "min_fake_probability": float(np.min(fake_probs)) if num_samples else None,
        "max_fake_probability": float(np.max(fake_probs)) if num_samples else None,
        "data_root": str(DATA_ROOT),
        "adapter_dir": str(ADAPTER_DIR),
        "random_seed": RANDOM_SEED,
        "max_images": MAX_IMAGES,
        "candidate_texts": CANDIDATE_TEXTS,
        "normalize_candidate_logprob": NORMALIZE_CANDIDATE_LOGPROB,
    }


metrics = summarize_real_only(pred_df, threshold=0.5)
threshold_df = pd.DataFrame([summarize_real_only(pred_df, threshold=t) for t in THRESHOLDS])

save_json(METRICS_DIR / "raise_real_metrics.json", metrics)
threshold_df.to_csv(METRICS_DIR / "raise_real_threshold_sweep.csv", index=False)
pred_df.groupby("extension").size().reset_index(name="num_samples").to_csv(METRICS_DIR / "raise_pred_extension_counts.csv", index=False)
pred_df.groupby("parent_folder").size().reset_index(name="num_samples").to_csv(METRICS_DIR / "raise_pred_parent_folder_counts.csv", index=False)

print(json.dumps(metrics, indent=2, ensure_ascii=False))
display(threshold_df)


In [ ]:
# Visualize the real images with the highest fake probability.
import matplotlib.pyplot as plt

TOP_K = 8
top_df = pred_df.sort_values("fake_probability", ascending=False).head(TOP_K)
cols = 4
rows = max(1, int(np.ceil(len(top_df) / cols)))

plt.figure(figsize=(4 * cols, 4 * rows))
for i, (_, row) in enumerate(top_df.iterrows(), start=1):
    image = load_rgb_image(Path(row["image_path"]))
    plt.subplot(rows, cols, i)
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"fake_p={row['fake_probability']:.3f}\n{row['file_name']}", fontsize=9)
plt.tight_layout()
plt.show()
